# Ajout du temps aux triptyques

Ce notebook remplace l'idée de deux notebooks séparés.

Il fait toujours la même chose : rattacher un `TIME` à chaque triptyque avec la même méthode UD.  
La seule différence est la source des temps :

- `coref` : les dates viennent de `COREF_name` dans le fichier `.entities` manuel ;
- `auto` : les dates viennent du CSV produit par la normalisation automatique des `TIME`.

L'intérêt est de comparer deux sources temporelles avec une méthode de rattachement identique.

In [21]:
import re
import ast
import bisect
import pandas as pd
import numpy as np

In [22]:
# Choisir la source des temps : "coref" ou "auto".
MODE_TEMPS = "coref"

DATE_DEBUT_CALENDRIER = "1872-10-02"

if MODE_TEMPS == "coref":
    TRIPTYQUES_CSV = "../results/csv_triptyques/manuel_chap1to5_chap.csv"
    ENTITIES_CSV = "../data/SACR/all_annots.sacr.entities"
    TOKENS_CSV = "../data/SACR/all_annots.sacr.tokens"
    OUT_CSV = "../results/csv_triptyques/manuel_chap1to5_chap_temps.csv"

if MODE_TEMPS == "auto":
    TRIPTYQUES_CSV = "../results/csv_triptyques/auto_all_chap.csv"
    ENTITIES_CSV = "../data/PROPP/all_txt/tdm_auto_allchap.entities"
    TOKENS_CSV = "../data/PROPP/all_txt/tdm_auto_allchap.tokens"
    TIME_CSV = "../results/csv_triptyques/time_mentions_auto.csv"
    OUT_CSV = "../results/csv_triptyques/auto_all_chap_temps.csv"

## 1. Charger les fichiers

On charge les triptyques, les entités et les tokens.  
Les tokens sont utiles parce que les triptyques utilisent souvent des identifiants locaux, alors que les entités `TIME` utilisent des identifiants globaux.

In [23]:
trip = pd.read_csv(TRIPTYQUES_CSV)
entities = pd.read_csv(ENTITIES_CSV, sep="\t")
tokens = pd.read_csv(TOKENS_CSV, sep="\t")

# On enlève les anciennes colonnes temporelles avant de recalculer.
time_cols = [c for c in trip.columns if c.startswith("time_")]
trip_base = trip.drop(columns=time_cols)

# On garde uniquement les entités temporelles.
times_ent = (
    entities[entities["cat"].eq("TIME")]
    .reset_index(drop=True)
    .reset_index()
    .rename(columns={"index": "entity_id"})
)

print("Mode :", MODE_TEMPS)
print("Triptyques :", trip_base.shape)
print("TIME :", times_ent.shape)

Mode : coref
Triptyques : (1255, 33)
TIME : (106, 34)


## 2. Préparer la table des temps

Cette partie change selon le mode choisi.

En mode `coref`, on lit directement le code contenu dans `COREF_name`.  
En mode `auto`, on lit le CSV de normalisation temporelle produit avant.

In [24]:
def parse_coref_code(code, date_debut=DATE_DEBUT_CALENDRIER):
    # Lit un code du type 1872-10-02-11 ou 1872_10_02_11.
    parts = re.split(r"[-_]", str(code).strip())
    nums = []

    # On garde au maximum année, mois, jour, heure, minute.
    for p in parts[:5]:
        nums.append(int(p) if str(p).isdigit() else 0)

    nums += [0] * (5 - len(nums))
    annee, mois, jour, heure, minute = nums[:5]

    out = {
        "time_kind": np.nan,
        "precision": np.nan,
        "datetime_auto": np.nan,
        "time_code_auto": np.nan,
        "jour_romanesque": np.nan,
        "debut_auto": np.nan,
        "fin_auto": np.nan,
        "duration_value": np.nan,
        "duration_unit": np.nan,
        "duration_relation": np.nan,
        "regle": np.nan,
    }

    # Sans année exploitable, on ne garde pas la mention comme date.
    if annee <= 0:
        return out

    out["time_kind"] = "date_absolue"
    out["regle"] = "coref"

    # Année seule.
    if mois <= 0:
        out["precision"] = "annee"
        out["time_code_auto"] = f"{annee:04d}-00-00-00-00"
        return out

    # Année + mois.
    if jour <= 0:
        out["precision"] = "mois"
        out["time_code_auto"] = f"{annee:04d}-{mois:02d}-00-00-00"
        return out

    # Date complète, éventuellement avec heure et minute.
    try:
        dt = pd.Timestamp(
            year=annee,
            month=mois,
            day=jour,
            hour=max(0, heure),
            minute=max(0, minute),
        )
    except Exception:
        return out

    out["precision"] = "minute" if minute else ("heure" if heure else "jour")
    out["datetime_auto"] = dt.strftime("%Y-%m-%d %H:%M")
    out["time_code_auto"] = f"{annee:04d}-{mois:02d}-{jour:02d}-{heure:02d}-{minute:02d}"
    out["jour_romanesque"] = (dt.normalize() - pd.Timestamp(date_debut)).days + 1

    return out


def build_times_from_coref(times_ent):
    # Les annotations manuelles portent le temps absolu dans COREF_name.
    if "COREF_name" not in times_ent.columns:
        raise ValueError("Le mode coref nécessite une colonne COREF_name dans le fichier .entities.")

    parsed = pd.DataFrame([
        parse_coref_code(code)
        for code in times_ent["COREF_name"]
    ])

    return pd.concat([times_ent.reset_index(drop=True), parsed], axis=1)


def build_times_from_auto_csv(times_ent, time_csv):
    # Le CSV auto contient déjà les mentions TIME traduites en dates, heures ou durées.
    times_norm = pd.read_csv(time_csv)

    wanted_cols = [
        "entity_id", "time_kind", "precision", "datetime_auto", "time_code_auto",
        "jour_romanesque", "debut_auto", "fin_auto", "duration_value",
        "duration_unit", "duration_relation", "regle"
    ]

    # On garde seulement les colonnes vraiment utilisées dans la suite.
    cols = [c for c in wanted_cols if c in times_norm.columns]

    return times_ent.merge(
        times_norm[cols],
        on="entity_id",
        how="left",
    )


if MODE_TEMPS == "coref":
    times = build_times_from_coref(times_ent)

if MODE_TEMPS == "auto":
    times = build_times_from_auto_csv(times_ent, TIME_CSV)

times.head()

,entity_id,COREF_name,COREF,start_token,end_token,cat,sacr_text,byte_onset,byte_offset,FUNCT,...,precision,datetime_auto,time_code_auto,jour_romanesque,debut_auto,fin_auto,duration_value,duration_unit,duration_relation,regle
0,0,1872-00-00-00,125,1,3,TIME,l'année 1872,3,15,NaN,...,annee,NaN,1872-00-00-00-00,NaN,NaN,NaN,NaN,NaN,NaN,coref
1,1,1814-00-00-00,123,24,25,TIME,en 1814,121,128,NaN,...,annee,NaN,1814-00-00-00-00,NaN,NaN,NaN,NaN,NaN,NaN,coref
2,2,1862-00-00-00,124,762,765,TIME,depuis de longues années,3775,3799,NaN,...,annee,NaN,1862-00-00-00-00,NaN,NaN,NaN,NaN,NaN,NaN,coref
3,3,1872-00-00-10,135,803,804,TIME,chaque jour,3980,3991,VA verbal_adjunct,...,annee,NaN,1872-00-00-00-00,NaN,NaN,NaN,NaN,NaN,NaN,coref
4,4,1872-00-00-12-17,137,1004,1008,TIME,à des heures chronométriquement déterminées,4948,4991,NaN,...,annee,NaN,1872-00-00-00-00,NaN,NaN,NaN,NaN,NaN,NaN,coref


## 3. Préparer les identifiants UD

On prépare deux dictionnaires :

- un accès aux tokens par identifiant global ;
- un accès aux tokens par position locale dans la phrase.

Cela permet de comparer correctement les verbes des triptyques avec les gouverneurs syntaxiques des `TIME`.

In [25]:
tok_global = {
    int(r.token_ID_within_document): r
    for r in tokens.itertuples()
}

tok_local = {
    (int(r.paragraph_ID), int(r.sentence_ID), int(r.token_ID_within_sentence)): r
    for r in tokens.itertuples()
}


def token_from_global(doc_id):
    # Cherche un token par son identifiant dans tout le document.
    if pd.isna(doc_id):
        return None
    return tok_global.get(int(doc_id))


def token_from_local(paragraph, sentence, local_id):
    # Cherche un token par sa position dans une phrase.
    if pd.isna(local_id):
        return None
    return tok_local.get((int(paragraph), int(sentence), int(local_id)))


def add_time_governor(row):
    # Dans .entities, head_syntactic_head_ID est un identifiant global.
    gov = token_from_global(row["head_syntactic_head_ID"])

    if gov is None:
        return pd.Series({
            "time_governor_global": np.nan,
            "time_governor_word": np.nan,
            "time_governor_dep": np.nan,
        })

    return pd.Series({
        "time_governor_global": int(gov.token_ID_within_document),
        "time_governor_word": gov.word,
        "time_governor_dep": gov.dependency_relation,
    })


times = pd.concat([times, times.apply(add_time_governor, axis=1)], axis=1)

# Clé de position dans le texte.
times["order_key"] = list(zip(
    times["paragraph_ID"].astype(int),
    times["sentence_ID"].astype(int),
    times["start_token"].astype(int),
))

## 4. Fonctions de conversion

Les triptyques contiennent souvent des identifiants locaux de tokens.  
On les convertit en identifiants globaux pour pouvoir les comparer aux entités `TIME`.

In [26]:
def safe_list(x):
    # Les listes sont souvent stockées sous forme de chaînes.
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x

    try:
        value = ast.literal_eval(str(x))
        return value if isinstance(value, list) else []
    except Exception:
        return []


def docid_from_row(row, col):
    # Utilise directement les DocID quand ils existent.
    if col in row.index and not pd.isna(row.get(col)):
        try:
            return int(row.get(col))
        except Exception:
            return np.nan
    return np.nan


def local_ids_to_global(paragraph, sentence, values):
    # Convertit les IDs locaux d'une cellule en IDs globaux.
    ids = set()

    for value in safe_list(values):
        try:
            local_id = int(value) - 1
            tok = token_from_local(paragraph, sentence, local_id)

            if tok is not None:
                ids.add(int(tok.token_ID_within_document))
        except Exception:
            pass

    return ids


def verb_global_id(row):
    # Récupère l'identifiant global du verbe du triptyque.
    docid = docid_from_row(row, "DocID_verbe")
    if not pd.isna(docid):
        return docid

    if pd.isna(row.get("ID_verbe")):
        return np.nan

    tok = token_from_local(
        row["Num_paragr"],
        row["Num_phrase"],
        int(row["ID_verbe"]) - 1,
    )

    return np.nan if tok is None else int(tok.token_ID_within_document)


def sujet_global_ids(row):
    # Récupère les tokens globaux du sujet.
    ids = set()

    docid = docid_from_row(row, "DocID_sujet")
    if not pd.isna(docid):
        ids.add(int(docid))

    ids |= local_ids_to_global(row["Num_paragr"], row["Num_phrase"], row.get("IDs_sujet"))
    return ids


def objet_global_ids(row):
    # Récupère les tokens globaux de l'objet.
    ids = set()

    docid = docid_from_row(row, "DocID_objet")
    if not pd.isna(docid):
        ids.add(int(docid))

    ids |= local_ids_to_global(row["Num_paragr"], row["Num_phrase"], row.get("IDs_objet"))
    return ids

## 5. Ancres temporelles

Un `TIME` local ne doit pas forcément devenir le temps de tout ce qui suit.

Par exemple, `à minuit précis` concerne localement un verbe.  
En revanche, `en l'année 1872` peut servir de cadre narratif.

In [27]:
ancestor_cache = {}


def ancestors_global(token_id):
    # Remonte les gouverneurs UD d'un token.
    if pd.isna(token_id):
        return []

    token_id = int(token_id)

    if token_id in ancestor_cache:
        return ancestor_cache[token_id]

    out = []
    seen = set()
    current = token_id

    while True:
        if current in seen:
            break

        seen.add(current)
        tok = token_from_global(current)

        if tok is None:
            break

        head = tok.syntactic_head_ID

        if pd.isna(head):
            break

        head = int(head)

        if head == current:
            break

        out.append(head)
        current = head

    ancestor_cache[token_id] = out
    return out


def is_descendant_of(child_id, ancestor_id):
    # Dit si un token dépend directement ou indirectement d'un autre token.
    if pd.isna(child_id) or pd.isna(ancestor_id):
        return False
    return int(ancestor_id) in ancestors_global(int(child_id))


def time_span_global(time_row):
    # Ensemble des tokens couverts par la mention TIME.
    return set(range(int(time_row["start_token"]), int(time_row["end_token"]) + 1))


BAD_ANCHOR_KINDS = {
    "duree",
    "frequence",
    "moment_vague",
    "garbage_probable",
    "non_parse",
    "reference_evenement",
    "relatif_non_calcule",
}

MAIN_DEPS = {"ROOT", "root", "conj", "parataxis"}


def has_time_value(row):
    # Un temps peut être une date ponctuelle ou un intervalle.
    return (
        not pd.isna(row.get("time_code_auto"))
        or not pd.isna(row.get("datetime_auto"))
        or not pd.isna(row.get("debut_auto"))
        or not pd.isna(row.get("fin_auto"))
    )


def is_narrative_anchor(row):
    # Une ancre narrative doit être calculable et portée par une proposition principale.
    if not has_time_value(row):
        return False

    if row.get("time_kind") in BAD_ANCHOR_KINDS:
        return False

    if str(row.get("regle")) == "il_y_a":
        return False

    return str(row.get("time_governor_dep")) in MAIN_DEPS


times["is_narrative_anchor"] = times.apply(is_narrative_anchor, axis=1)

# Accès rapide aux TIME d'une phrase.
time_groups = {
    key: group
    for key, group in times.groupby(["paragraph_ID", "sentence_ID"])
}

# Liste ordonnée des temps qui peuvent servir de contexte narratif.
anchor_times = (
    times[times["is_narrative_anchor"]]
    .sort_values("order_key")
    .reset_index(drop=True)
)

anchor_keys = anchor_times["order_key"].tolist()

## 6. Choisir un temps pour chaque triptyque

La règle est la même dans les deux modes :

1. si le `TIME` est dans l'objet, on le prend ;
2. sinon s'il est dans le sujet, on le prend ;
3. sinon s'il dépend directement du verbe en UD, on le prend ;
4. sinon, si c'est une vraie ancre de cadrage, elle peut couvrir un verbe dépendant ;
5. sinon, on reprend le dernier temps de cadrage connu.

In [28]:
def triplet_order_key(row):
    # Position approximative du triptyque dans le texte.
    verb_id = verb_global_id(row)

    if pd.isna(verb_id):
        verb_id = 10**12

    return (
        int(row["Num_paragr"]),
        int(row["Num_phrase"]),
        int(verb_id),
    )


def candidate_rank(trip_row, time_row):
    # Classe les TIME possibles pour un triptyque donné.
    verb_id = verb_global_id(trip_row)

    if pd.isna(verb_id):
        return -999, "aucun"

    time_span = time_span_global(time_row)
    obj_ids = objet_global_ids(trip_row)
    subj_ids = sujet_global_ids(trip_row)
    gov_id = time_row.get("time_governor_global")

    # Le TIME est directement dans l'objet du triptyque.
    if obj_ids and time_span.intersection(obj_ids):
        return 120, "objet"

    # Le TIME est directement dans le sujet du triptyque.
    if subj_ids and time_span.intersection(subj_ids):
        return 110, "sujet"

    # Le TIME dépend directement du verbe du triptyque.
    if not pd.isna(gov_id) and int(gov_id) == int(verb_id):
        return 100, "ud_direct"

    # Portée large seulement pour les vrais temps de cadrage.
    if bool(time_row.get("is_narrative_anchor")) and not pd.isna(gov_id):
        if is_descendant_of(verb_id, gov_id):
            return 80, "ud_scope_anchor"

    return -999, "aucun"


def choose_time_same_sentence(trip_row):
    # Cherche le meilleur TIME dans la phrase du triptyque.
    key = (int(trip_row["Num_paragr"]), int(trip_row["Num_phrase"]))
    candidates = time_groups.get(key)

    if candidates is None or candidates.empty:
        return None, "aucun"

    verb_id = verb_global_id(trip_row)
    ranked = []

    for _, time_row in candidates.iterrows():
        rank, source = candidate_rank(trip_row, time_row)

        if rank > -999:
            distance = 999999 if pd.isna(verb_id) else abs(int(time_row["start_token"]) - int(verb_id))
            ranked.append((rank, -distance, source, time_row))

    if not ranked:
        return None, "aucun"

    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)
    return ranked[0][3], ranked[0][2]


def previous_anchor(trip_row):
    # Récupère le dernier temps narratif solide avant le triptyque.
    key = triplet_order_key(trip_row)
    position = bisect.bisect_left(anchor_keys, key) - 1

    if position < 0:
        return None

    return anchor_times.iloc[position]


def empty_time_row():
    # Cas sans temps trouvé.
    return pd.Series({
        "time_entity_id": np.nan,
        "time_texte": np.nan,
        "time_kind": np.nan,
        "time_precision": np.nan,
        "time_code": np.nan,
        "time_datetime": np.nan,
        "time_debut": np.nan,
        "time_fin": np.nan,
        "time_jour_romanesque": np.nan,
        "time_duration_value": np.nan,
        "time_duration_unit": np.nan,
        "time_duration_relation": np.nan,
        "time_source": "aucun",
        "time_ud_governor": np.nan,
    })


def assign_time(trip_row):
    # 1. Chercher un TIME localement lié au triptyque.
    time_row, source = choose_time_same_sentence(trip_row)

    # 2. Sinon, reprendre le dernier temps de cadrage.
    if time_row is None:
        time_row = previous_anchor(trip_row)
        source = "precedent_context" if time_row is not None else "aucun"

    if time_row is None:
        return empty_time_row()

    return pd.Series({
        "time_entity_id": int(time_row["entity_id"]),
        "time_texte": time_row["text"],
        "time_kind": time_row.get("time_kind"),
        "time_precision": time_row.get("precision"),
        "time_code": time_row.get("time_code_auto"),
        "time_datetime": time_row.get("datetime_auto"),
        "time_debut": time_row.get("debut_auto"),
        "time_fin": time_row.get("fin_auto"),
        "time_jour_romanesque": time_row.get("jour_romanesque"),
        "time_duration_value": time_row.get("duration_value"),
        "time_duration_unit": time_row.get("duration_unit"),
        "time_duration_relation": time_row.get("duration_relation"),
        "time_source": source,
        "time_ud_governor": time_row.get("time_governor_word"),
    })

## 7. Export

Le fichier final garde les triptyques et ajoute seulement les colonnes temporelles utiles.

In [29]:
assigned = trip_base.apply(assign_time, axis=1)
out = pd.concat([trip_base, assigned], axis=1)

out.to_csv(OUT_CSV, index=False, encoding="utf-8")

print("Fichier écrit :", OUT_CSV)
print("Triptyques :", len(out))
print("Triptyques avec temps :", out["time_code"].notna().sum() + out["time_debut"].notna().sum())
out["time_source"].value_counts(dropna=False)

Fichier écrit : ../results/csv_triptyques/manuel_chap1to5_chap_temps.csv
Triptyques : 1255
Triptyques avec temps : 1218


time_source
precedent_context    1058
objet                  70
ud_scope_anchor        68
ud_direct              52
sujet                   7
Name: count, dtype: int64

In [30]:
# Contrôle rapide sur les cas de rattachement.
out[[
    "Phrase", "Sujet", "Verbe", "Objet",
    "time_texte", "time_code", "time_source", "time_ud_governor"
]].head(20)

,Phrase,Sujet,Verbe,Objet,time_texte,time_code,time_source,time_ud_governor
0,"En l' année 1872, la maison portant le numéro ...",NaN,portant,le numéro 7 de Saville-row Burlington Gardens-...,l' année 1872,1872-00-00-00-00,ud_scope_anchor,habitée
1,"En l' année 1872, la maison portant le numéro ...",Sheridan,mourut,en 1814,en 1814,1814-00-00-00-00,objet,mourut
2,"En l' année 1872, la maison portant le numéro ...",la maison,habitée,En l'année 1872,l' année 1872,1872-00-00-00-00,objet,habitée
3,"En l' année 1872, la maison portant le numéro ...",la maison,habitée,par Phileas Fogg esq.,l' année 1872,1872-00-00-00-00,ud_direct,habitée
4,"En l' année 1872, la maison portant le numéro ...",la maison,habitée,l'un des membres les plus singuliers et les pl...,l' année 1872,1872-00-00-00-00,ud_direct,habitée
5,"En l' année 1872, la maison portant le numéro ...",la maison,semblât,NaN,l' année 1872,1872-00-00-00-00,ud_scope_anchor,habitée
6,"En l' année 1872, la maison portant le numéro ...",NaN,prendre,à tâche,l' année 1872,1872-00-00-00-00,ud_scope_anchor,habitée
7,"En l' année 1872, la maison portant le numéro ...",NaN,faire,NaN,l' année 1872,1872-00-00-00-00,ud_scope_anchor,habitée
8,"En l' année 1872, la maison portant le numéro ...",qui,pût,NaN,l' année 1872,1872-00-00-00-00,ud_scope_anchor,habitée
9,"En l' année 1872, la maison portant le numéro ...",qui,attirer,l'attention,l' année 1872,1872-00-00-00-00,ud_scope_anchor,habitée
